# FOMO — SJSU Headcount Training & Visualization on Combined Scenes

End-to-end notebook that:
1. Clones [fomo-edge-ai/fomo](https://github.com/fomo-edge-ai/fomo) and installs it
2. Downloads and combines the **SJSU Headcount Scene-1, Scene-2, and Scene-3** datasets from HuggingFace (`bdanko/sjsu-headcount-scene-1`, `-2`, `-3`)
3. Trains **FOMO-m** (the lightweight MobileNetV2 point-localization model) for 10 epochs
4. Visualizes the predictions on validation images — predicted person locations as coloured circles overlaid on the original image

## 1 — Environment Setup

In [ ]:
!pip install fomo-edge-ai==0.0.11

You must restart the session after running the above

In [ ]:

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


## 2 — Download and Combine Datasets

We download the three datasets from HuggingFace (`bdanko/sjsu-headcount-scene-1`, `bdanko/sjsu-headcount-scene-2`, and `bdanko/sjsu-headcount-scene-3`).
We merge their splits (train, validation, test) into a combined directory, prefixing filenames to prevent name conflicts.

In [ ]:
import shutil
import zipfile
from pathlib import Path
import yaml
from huggingface_hub import hf_hub_download

HF_REPOS = {
    "scene-1": "bdanko/sjsu-headcount-scene-1",
    "scene-2": "bdanko/sjsu-headcount-scene-2",
    "scene-3": "bdanko/sjsu-headcount-scene-3"
}

RAW_ROOT = Path("raw_datasets")
DATASET_ROOT = Path("sjsu-headcount-combined")

# 1. Download and extract all dataset zips from scratch
for prefix, repo_id in HF_REPOS.items():
    dataset_dir = RAW_ROOT / prefix
    zip_name = f"sjsu-headcount-scene-{prefix.split('-')[-1]}.zip"
    print(f"Downloading {zip_name} from {repo_id}...")
    shutil.rmtree(dataset_dir, ignore_errors=True)
    zip_path = hf_hub_download(
        repo_id=repo_id,
        filename=zip_name,
        repo_type="dataset"
    )
    print(f"Extracting to {dataset_dir}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print(f"Dataset {prefix} extraction complete")

# Clean target combined dataset folder
shutil.rmtree(DATASET_ROOT, ignore_errors=True)

# 2. Create combined directory structure
for split in ["train", "valid", "test"]:
    (DATASET_ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / split / "labels").mkdir(parents=True, exist_ok=True)

# 3. Merge splits and prefix filenames to avoid collisions
print("\nCombining datasets...")
for prefix, repo_id in HF_REPOS.items():
    dataset_dir = RAW_ROOT / prefix
    
    split_mapping = {
        "train": "train",
        "valid": "valid",
        "test": "test"
    }
    
    for src_split, dst_split in split_mapping.items():
        src_img_dir = dataset_dir / src_split / "images"
        src_lbl_dir = dataset_dir / src_split / "labels"
        
        if not src_img_dir.exists() or not src_lbl_dir.exists():
            continue
            
        dst_img_dir = DATASET_ROOT / dst_split / "images"
        dst_lbl_dir = DATASET_ROOT / dst_split / "labels"
        
        for img_path in src_img_dir.glob("*"):
            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                new_name = f"{prefix}_{img_path.name}"
                shutil.copy2(img_path, dst_img_dir / new_name)
                
                # Corresponding label file
                lbl_name = f"{img_path.stem}.txt"
                lbl_path = src_lbl_dir / lbl_name
                if lbl_path.exists():
                    shutil.copy2(lbl_path, dst_lbl_dir / f"{prefix}_{lbl_name}")

print("Dataset combination complete.")

# 4. Create data.yaml for combined dataset
data_yaml_path = DATASET_ROOT / "data.yaml"
data_cfg = {
    "path": str(DATASET_ROOT.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["person"]
}
data_yaml_path.write_text(yaml.dump(data_cfg, default_flow_style=False))

print(f"\nDataset config:")
print(f"  path : {data_cfg['path']}")
print(f"  train: {data_cfg['train']}")
print(f"  val  : {data_cfg['val']}")
print(f"  nc   : {data_cfg['nc']} ({data_cfg['names']})")

# Quick sanity check — count images and labels
def count_split(split_dir):
    imgs = list(split_dir.rglob("*.jpg")) + list(split_dir.rglob("*.png")) + list(split_dir.rglob("*.jpeg"))
    lbl_dir = split_dir.parent / "labels"
    lbls = list(lbl_dir.rglob("*.txt")) if lbl_dir.exists() else []
    return len(imgs), len(lbls)

train_imgs, train_lbls = count_split(DATASET_ROOT / "train" / "images")
val_imgs, val_lbls     = count_split(DATASET_ROOT / "valid" / "images")
test_imgs, test_lbls   = count_split(DATASET_ROOT / "test" / "images")
print(f"Combined Train : {train_imgs} images, {train_lbls} label files")
print(f"Combined Val   : {val_imgs} images, {val_lbls} label files")
print(f"Combined Test  : {test_imgs} images, {test_lbls} label files")



## 3 — Train FOMO-m

FOMO-m uses a **MobileNetV2 backbone (α=0.5)** with a single-pixel detection head.
Input resolution is **192** → output grid **24x24** (8× downsample).

Training uses:
- Adam, lr=3e-4, ReduceLROnPlateau on val F1
- Weighted CrossEntropy (background=1×, foreground=100×)
- No augmentation (plain resize + ImageNet normalisation)
- Val sweep over confidence thresholds every epoch

In [ ]:
from fomo import FOMO

EPOCHS     = 100
BATCH      = 16
MODEL_SIZE = "m"            # "s" | "m" | "l"
PROJECT    = "runs/fomo"
RUN_NAME   = f"sjsu_headcount_all_scenes_{MODEL_SIZE}"

model = FOMO(model_path=None, size=MODEL_SIZE, nb_classes=1, device=device)
print(f"Model: FOMO-{MODEL_SIZE}")
print(f"  Input size : {model.input_size}×{model.input_size}")
print(f"  Grid size  : {model.input_size//8}×{model.input_size//8}")
print(f"  Classes    : {model.nb_classes}")

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

results = model.train(
    allow_experimental=True,
    data=str(data_yaml_path),
    epochs=EPOCHS,
    batch=BATCH,
    lr0=3e-4,
    eval_interval=1,
    workers=2,
    device=device,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    
    # Early stopping
    patience=12,
    
    # Regularization & Speedups
    optimizer="adamw",
    weight_decay=1e-4,
    ema=True,
    amp=True,
    
    # Data Augmentations
    mosaic_prob=1.0,
    flip_prob=0.5,
    hsv_prob=1.0,
)

save_dir = Path(results["save_dir"])
print(f"\nTraining complete. Saved to: {save_dir}")


In [ ]:

from fomo import FOMO

weights_dir = save_dir / "weights"
best_pt  = weights_dir / "best.pt"
last_pt  = weights_dir / "last.pt"
ckpt_path = best_pt if best_pt.exists() else last_pt

trained = FOMO(str(ckpt_path), device=device)
print(f"Loaded: {ckpt_path.name}")
print(f"  family : {trained.family}")
print(f"  size   : {trained.size}")
print(f"  imgsz  : {trained.input_size}")
print(f"  nc     : {trained.nb_classes}")


## 6 — Export to TFLite (FP32)

We can now export the trained PyTorch model to a TFLite FP32 flatbuffer using the `export` API.

In [ ]:

fp32_path = trained.export(output_path=str(weights_dir / f"{RUN_NAME}_fp32.tflite"))
print(f"FP32 TFLite model exported to: {fp32_path}")


## 7 — INT8 Quantization

To deploy the model on edge devices (like the Arm Ethos-U NPU), we quantize the weights and activations to INT8.
This uses `INT8Quantizer` with a representative calibration dataset from the validation set.

In [ ]:

import numpy as np
from PIL import Image
from torchvision import transforms
from fomo.export import INT8Quantizer, INT8QuantizeConfig

val_img_dir = DATASET_ROOT / "valid" / "images"
calibration_images = sorted(val_img_dir.rglob("*.jpg"))

image_transform = transforms.Compose([
    transforms.Resize((trained.input_size, trained.input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

def get_calibration_data(image_paths, num_samples=150):
    """Yields representative data formatted for the LiteRT graph."""
    count = 0
    for img_path in image_paths:
        if count >= num_samples:
            return
        img = Image.open(img_path).convert("RGB")
        tensor = image_transform(img).unsqueeze(0).numpy()
        yield {"args_0": tensor}
        count += 1

num_samples = min(len(calibration_images), 150)
print(f"Using {num_samples} validation images for calibration.")
calib_iter = get_calibration_data(calibration_images, num_samples=num_samples)

config = INT8QuantizeConfig(num_calibration_samples=num_samples)
quantizer = INT8Quantizer()

int8_path = quantizer.quantize(
    fp32_tflite=fp32_path,
    calibration_data=calib_iter,
    config=config,
    output_path=str(weights_dir / f"{RUN_NAME}_int8.tflite")
)
print(f"INT8 quantized model exported to: {int8_path}")


## Vela Compilation

In [ ]:
!pip install -q ethos-u-vela

In [ ]:
!wget https://raw.githubusercontent.com/bencejdanko/Overhead-People-Counting-YOLOXNano-FOMO-Ethos-U55-NPU/refs/heads/main/configs/default_vela.ini

!vela /content/runs/fomo/sjsu_headcount_all_scenes_m/weights/sjsu_headcount_all_scenes_m_int8.tflite \
  --accelerator-config ethos-u55-256 \
  --config /content/default_vela.ini \
  --memory-mode Shared_Sram \
  --system-config Ethos_U55_High_End_Embedded \
  --output-dir vela_output \
  --optimise Performance
